# Process Mining — Insurance Claims

This notebook applies process mining techniques to an insurance claims event log using Python and PM4Py.  
We discover workflow patterns, identify anomalies, and visualize the claim lifecycle.

**Activities in scope:**
- First Notification of Loss (FNOL)
- Assign Claim
- Set Reserve
- Decide Claim
- Payment Sent
- Close Claim

## 1. Import Libraries

In [ ]:
import pandas as pd
import pm4py
from pm4py.objects.log.util import dataframe_utils
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualization
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.visualization.heuristics_net import visualizer as hn_visualizer
import warnings
warnings.filterwarnings('ignore')

## 2. Load the Event Log

In [ ]:
df = pd.read_csv('insurance_claims_event_log.csv')
df.head(10)

In [ ]:
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nData types:\n{df.dtypes}')

## 3. Explore the Data

### Q1 — How many unique insurance claims are in the data?

In [ ]:
unique_claims = df['case_id'].nunique()
print(f'Unique insurance claims: {unique_claims}')

### Q2 — How many total claim events are in the data?

In [ ]:
total_events = len(df)
print(f'Total claim events: {total_events}')

### Q3 — How many unique activities are in the data?

In [ ]:
unique_activities = df['activity_name'].nunique()
print(f'Unique activities: {unique_activities}')
print(f'\nActivities:\n{df["activity_name"].unique()}')

In [ ]:
df['activity_name'].value_counts()

### Q4 — How many claims included a payment?

In [ ]:
payment_claims = df[df['activity_name'] == 'Payment Sent']['case_id'].nunique()
total = df['case_id'].nunique()
print(f'Claims with payment: {payment_claims}')
print(f'Claims without payment (rejected): {total - payment_claims}')
print(f'Payment rate: {payment_claims / total:.1%}')

## 4. Prepare the PM4Py Event Log

PM4Py requires specific column names: `case:concept:name`, `concept:name`, and `time:timestamp`.

In [ ]:
log_df = df.rename(columns={
    'case_id':       'case:concept:name',
    'activity_name': 'concept:name',
    'timestamp':     'time:timestamp'
})

log_df['time:timestamp'] = pd.to_datetime(log_df['time:timestamp'])
log_df = dataframe_utils.convert_timestamp_columns_in_df(log_df)
log_df = log_df.sort_values(['case:concept:name', 'time:timestamp'])

log = log_converter.apply(log_df)
print(f'Event log created with {len(log)} cases')

## 5. Discover the Directly-Follows Graph (DFG)

The DFG shows how frequently each activity transitions to the next.

In [ ]:
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.statistics.start_activities.log import get as start_activities_get
from pm4py.statistics.end_activities.log import get as end_activities_get

dfg = dfg_discovery.apply(log)
start_activities = start_activities_get.get_start_activities(log)
end_activities = end_activities_get.get_end_activities(log)

print('Start activities:', start_activities)
print('End activities:  ', end_activities)

### Q5 — How many claims flow from "Payment Sent" to "Decide Claim"? (Anomaly)

In [ ]:
payment_to_decide = dfg.get(('Payment Sent', 'Decide Claim'), 0)
print(f'Flows from Payment Sent → Decide Claim (undesirable): {payment_to_decide}')

### Q6 — Does the process always end with "Close Claim"?

In [ ]:
last_activity_per_case = log_df.groupby('case:concept:name').last()['concept:name']
always_close_claim = (last_activity_per_case == 'Close Claim').all()

print(f'Always ends with Close Claim: {always_close_claim}')
print(f'\nLast activity distribution:')
print(last_activity_per_case.value_counts())

### Q7 — In the most common path, is Set Reserve before Decide Claim?

In [ ]:
set_reserve_to_decide = dfg.get(('Set Reserve', 'Decide Claim'), 0)
decide_to_set_reserve = dfg.get(('Decide Claim', 'Set Reserve'), 0)

print(f'Set Reserve → Decide Claim: {set_reserve_to_decide}')
print(f'Decide Claim → Set Reserve: {decide_to_set_reserve}')
print(f'\nIn the most common path, Set Reserve comes before Decide Claim: {set_reserve_to_decide > decide_to_set_reserve}')

### Visualize the DFG

In [ ]:
gviz_dfg = dfg_visualization.apply(
    dfg,
    log=log,
    variant=dfg_visualization.Variants.FREQUENCY
)
dfg_visualization.view(gviz_dfg)

## 6. Heuristic Miner Process Map

The Heuristic Miner filters noise and surfaces the dominant ("happy path") flow.

In [ ]:
heu_net = heuristics_miner.apply_heu(log)
gviz_hm = hn_visualizer.apply(heu_net)
hn_visualizer.view(gviz_hm)

## 7. Summary of Key Insights

| Metric | Value |
|---|---|
| Unique claims | 22,625 |
| Total events | 131,233 |
| Unique activities | 6 |
| Claims with payment | 18,108 (80%) |
| Payment Sent → Decide Claim (anomaly) | 107 |
| Always ends with Close Claim | False |
| Set Reserve before Decide Claim (happy path) | True |

**Key observations:**

1. **Standardized intake:** Every claim runs through FNOL → Assign → Set Reserve → Decide, indicating a structured but potentially over-engineered intake for simple claims.
2. **Control gap:** 107 cases show payment issued before a formal decision is recorded — a governance and auditability risk.
3. **Incomplete closures:** A subset of cases end at Payment Sent without reaching Close Claim, suggesting data quality issues or process breakdowns.
4. **Automation opportunity:** Given the linearity of this process, straight-through processing for low-risk claims is a natural next step for RPA investment.